In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Carregar dados
df = pd.read_csv('data/merge_final.csv')

# Selecionar e tratar variáveis
variaveis = [
    'tx_distorcao_fundamental',  # Já existe no DataFrame
    'taxa_escolas_por_habitante',  # Confirmar nome correto
    'ideb_ano_inicial',  # Confirmar correspondência com "Ideb_inicial"
    'Total_alfabetização', 
    'Taxa de abandonos EF'
]

# Converter 'Total_alfabetização' para float
df['Total_alfabetização'] = df['Total_alfabetização'].str.replace(',', '.').astype(float)

# Preencher valores NaN com a média da respectiva coluna
df[variaveis] = df[variaveis].fillna(df[variaveis].mean())

# Normalização
scaler = StandardScaler()
dados_normalizados = scaler.fit_transform(df[variaveis])

# Escolher K ideal com base no gráfico anterior
k_ideal = 3  # Pode mudar para 4 caso deseje mais precisão
kmeans = KMeans(n_clusters=k_ideal, random_state=42, n_init=10)
clusters = kmeans.fit_predict(dados_normalizados)

# Adicionar os clusters e valores normalizados ao DataFrame original
df['Cluster'] = clusters

# Criar um novo DataFrame apenas com as cidades, clusters e valores normalizados
df_final = df[['Cluster']].copy()
df_final[variaveis] = dados_normalizados  # Adiciona os valores normalizados
df_final.insert(0, 'Cidades', df['Cidades'])  # Supondo que a coluna de cidade se chama 'Cidade'

# Salvar em CSV
df_final.to_csv('cidades_clusterizadas.csv', index=False)

# Gerar médias de cada cluster
df_medias = df.groupby('Cluster')[variaveis].mean()

# Salvar as médias em CSV
df_medias.to_csv('medias_clusters.csv')

# Exibir amostras dos resultados
print("Cidades Clusterizadas:")
print(df_final.head())

print("\nMédias de cada Cluster:")
print(df_medias)


Cidades Clusterizadas:
              Cidades  Cluster  tx_distorcao_fundamental  \
0               Acauã        1                 -0.809321   
1        Agricolândia        1                 -0.266055   
2  Alagoinha do Piauí        2                 -0.795739   
3   Alegrete do Piauí        2                 -1.026627   
4          Alto Longá        2                 -0.822902   

   taxa_escolas_por_habitante  ideb_ano_inicial  Total_alfabetização  \
0                    0.091024          0.900797             1.237444   
1                   -0.896312         -0.243891             0.413441   
2                    0.589648         -0.129422            -2.090414   
3                    0.201145          0.442922            -1.214662   
4                    0.964496          0.213984            -1.023588   

   Taxa de abandonos EF  
0             -0.704940  
1             -0.170268  
2              0.631740  
3             -0.526716  
4             -0.615828  

Médias de cada Cluster:
  

In [1]:
import pandas as pd
import geopandas as gpd
import plotly.express as px
import numpy as np
from datetime import datetime

# -------------------------------
# 1. Carregar os dados gerados pelo KMeans
# -------------------------------
df_clusters = pd.read_csv('output/cidades_clusterizadas_educ.csv')

# Listar as colunas disponíveis
print("Colunas disponíveis:", df_clusters.columns.tolist())

# Verificar os valores únicos de cluster
print("\nValores únicos de cluster:", df_clusters['Cluster'].unique())
print(f"Total de municípios por cluster:")
print(df_clusters['Cluster'].value_counts().sort_index())

# Normalizar os nomes dos municípios para facilitar a junção com o shapefile
df_clusters['Cidades'] = df_clusters['Cidades'].str.upper().str.normalize('NFKD')\
    .str.encode('ascii', errors='ignore').str.decode('utf-8')

# Usar o nome correto da coluna de cluster (com C maiúsculo neste caso)
cluster_column = 'Cluster'

# Converter cluster para string para garantir compatibilidade na visualização
df_clusters[cluster_column] = df_clusters[cluster_column].astype(str)

# Criar um mapeamento para exibição dos clusters começando do 1 em vez do 0
cluster_display_map = {'0': '1', '1': '2', '2': '3'}
df_clusters['cluster_display'] = df_clusters[cluster_column].map(cluster_display_map)

# Descrições vazias para os clusters seguindo o modelo
cluster_descriptions = {
    '0': '',
    '1': '',
    '2': ''
}

# Verificar quais clusters realmente existem nos dados
existing_clusters = df_clusters[cluster_column].unique()
print("Clusters existentes nos dados:", existing_clusters)

# Garantir que todos os clusters têm uma descrição
for cluster in existing_clusters:
    if cluster not in cluster_descriptions:
        # Usar o mapeamento para exibir o número do cluster começando em 1
        display_num = cluster_display_map.get(cluster, str(int(cluster) + 1))
        cluster_descriptions[cluster] = f"Cluster {display_num}"

df_clusters['Cluster_Descricao'] = df_clusters[cluster_column].map(cluster_descriptions)

# -------------------------------
# 2. Carregar o shapefile dos municípios
# -------------------------------
shapefile_path = 'media/PI_Municipios_2022.shp'  # Ajuste o caminho conforme necessário
try:
    gdf = gpd.read_file(shapefile_path)
    print(f"Shapefile carregado com sucesso. Total de {len(gdf)} municípios.")
except Exception as e:
    print(f"Erro ao carregar shapefile: {e}")
    print("Verifique se o arquivo existe no caminho especificado.")
    exit(1)

# Normalizar os nomes dos municípios no shapefile
gdf['NM_MUN'] = gdf['NM_MUN'].str.upper().str.normalize('NFKD')\
    .str.encode('ascii', errors='ignore').str.decode('utf-8')

# -------------------------------
# 3. Mesclar os clusters com o GeoDataFrame
# -------------------------------
# Selecionar as colunas relevantes para o mapa
colunas_para_mapa = ['Cidades', cluster_column, 'Cluster_Descricao', 'cluster_display']

# Adicionar as colunas de indicadores para hover data
colunas_indicadores_educacao = [col for col in df_clusters.columns if col not in ['Cidades', 'Cluster', 'Cluster_Descricao', 'cluster_display']]
colunas_para_mapa.extend(colunas_indicadores_educacao)

# Realizar a fusão dos dados
gdf_merged = gdf.merge(df_clusters[colunas_para_mapa],
                      left_on='NM_MUN', right_on='Cidades', how='left')

# Verificar quantos municípios foram mesclados com sucesso
matched_count = gdf_merged[~gdf_merged[cluster_column].isna()].shape[0]
total_count = gdf_merged.shape[0]
print(f"Municípios mesclados com sucesso: {matched_count} de {total_count} ({matched_count/total_count*100:.1f}%)")

# Listar municípios que não foram encontrados na junção
if matched_count < total_count:
    unmatched = gdf_merged[gdf_merged[cluster_column].isna()]['NM_MUN'].tolist()
    print(f"Municípios não encontrados nos dados de cluster ({len(unmatched)}):")
    print(", ".join(unmatched[:10]) + ("..." if len(unmatched) > 10 else ""))

# -------------------------------
# 4. Criar um dicionário de cores para os clusters
# -------------------------------
# Cores personalizadas por cluster
custom_colors = {
    '0': '#d62728',  # Vermelho para Cluster 1
    '1': '#ff7f0e',  # Laranja para Cluster 2
    '2': '#2ca02c'   # Verde para Cluster 3
}

# Garantir que temos cores para todos os clusters
for cluster in existing_clusters:
    if cluster not in custom_colors:
        # Gerar uma cor aleatória para clusters não mapeados
        custom_colors[cluster] = f'#{np.random.randint(0, 16777215):06x}'

# -------------------------------
# 5. Preparar dados para hover
# -------------------------------
# Criar colunas formatadas para exibição no hover
hover_columns = {}

# Formatar cada indicador para o hover
for col in colunas_indicadores_educacao:
    if df_clusters[col].dtype in [np.float64, np.int64]:
        # Formatar valores numéricos
        if df_clusters[col].max() <= 1.0:  # Provavelmente é uma proporção
            gdf_merged[f'{col} (%)'] = gdf_merged[col].fillna(0).apply(lambda x: f"{x*100:.2f}%")
            hover_columns[f'{col} (%)'] = True
        else:  # É um valor absoluto ou percentual já formatado
            gdf_merged[f'{col}_fmt'] = gdf_merged[col].fillna(0).apply(lambda x: f"{x:.2f}")
            hover_columns[f'{col}_fmt'] = True
    elif df_clusters[col].dtype == bool or set(df_clusters[col].dropna().unique()).issubset({0, 1}):
        # Formatar valores booleanos
        gdf_merged[f'{col}_fmt'] = gdf_merged[col].fillna(0).apply(lambda x: "Sim" if x == 1 else "Não")
        hover_columns[f'{col}_fmt'] = True
    else:
        # Para outros tipos de dados, incluir diretamente
        hover_columns[col] = True

# Adicionar a descrição do cluster ao hover
hover_columns['Cluster_Descricao'] = True
# Usar o número do cluster para exibição (começando de 1)
hover_columns['cluster_display'] = True
# Ocultar o número original do cluster
hover_columns[cluster_column] = False

# -------------------------------
# 6. Criar o mapa interativo
# -------------------------------
try:
    fig = px.choropleth_mapbox(
        gdf_merged,
        geojson=gdf_merged.geometry,
        locations=gdf_merged.index,
        color=cluster_column,  # Continuamos usando o cluster original para manter a consistência das cores
        color_discrete_map=custom_colors,
        hover_name='NM_MUN',
        hover_data=hover_columns,
        mapbox_style="carto-positron",
        center={"lat": -7.718, "lon": -42.728},
        zoom=6,
        opacity=0.7,
        labels={
            'Cluster_Descricao': 'Perfil',
            'cluster_display': 'Cluster'
        }
    )

    # -------------------------------
    # 7. Ajustar a legenda e exibição
    # -------------------------------
    # Adicionar contagens de municípios por cluster no título das legendas
    legend_counts = df_clusters[cluster_column].value_counts().to_dict()
    legend_counts = {str(k): v for k, v in legend_counts.items()}  # Garantir que as chaves são strings
    
    # Criar títulos para legendas com contagens e números de cluster ajustados
    cluster_legend_labels = {}
    for cluster, description in cluster_descriptions.items():
        count = legend_counts.get(cluster, 0)
        # Usar o número mapeado do cluster (começando de 1)
        display_num = cluster_display_map.get(cluster, str(int(cluster) + 1))
        cluster_legend_labels[cluster] = f"Cluster {display_num} ({count} municípios)"
    
    # Atualizar layout com legendas melhoradas
    fig.update_layout(
        legend_title_text='Perfil Educacional dos Municípios',
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=0.01,
            itemsizing="constant"
        ),
        margin={"r": 0, "t": 30, "l": 0, "b": 0},
        title={
            'text': "Classificação de Municípios por Perfil Educacional - 3 Clusters",
            'y':0.97,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top'
        }
    )
    
    # Atualizar os nomes das legendas para incluir contagens e usar numeração a partir de 1
    for i, trace in enumerate(fig.data):
        cluster_id = trace.name
        if cluster_id in cluster_legend_labels:
            fig.data[i].name = cluster_legend_labels[cluster_id]

    # -------------------------------
    # 8. Adicionar resumo das características dos clusters
    # -------------------------------
    # Texto resumo das características dos clusters de educação
    summary_text = """<b>Análise dos Clusters Educacionais:</b><br>
    <b>Cluster 1</b>: Emergência Educacional - Baixos índices em todos os indicadores, situação crítica que exige ação imediata<br>
    <b>Cluster 2</b>: Pouca infraestrutura - Melhor que o Cluster 1, mas ainda com grandes deficiências de infraestrutura<br>
    <b>Cluster 3</b>: Contradição Educacional - Resultados acadêmicos contraditórios: alta evasão com bom desempenho em avaliações
    """
    
    # Adicionar como anotação de texto no mapa
    fig.add_annotation(
        xref="paper", yref="paper",
        x=0.5, y=0,
        text=summary_text,
        showarrow=False,
        font=dict(size=10),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="black",
        borderwidth=1,
        align="left",
        xanchor="center",
        yanchor="bottom"
    )
    
    # Salvar o mapa interativo com nome incluindo data
    date_str = datetime.now().strftime("%Y%m%d")
    output_file = f"mapa_interativo_3clusters_educacao_{date_str}.html"
    fig.write_html(output_file)
    print(f"Mapa interativo salvo como '{output_file}'")
    
    # No Jupyter ou ambiente que suporte exibição, você pode chamar fig.show()
    # fig.show()
    
except Exception as e:
    print(f"Erro ao criar o mapa: {e}")
    import traceback
    traceback.print_exc()
    print("Verifique se os dados estão formatados corretamente e se o GeoPandas está instalado.")

Colunas disponíveis: ['Cidades', 'Cluster', 'tx_distorcao_fundamental', 'taxa_escolas_por_habitante', 'ideb_ano_inicial', 'Total_alfabetização', 'Taxa de abandonos EF']

Valores únicos de cluster: [1 2 0]
Total de municípios por cluster:
Cluster
0    66
1    99
2    59
Name: count, dtype: int64
Clusters existentes nos dados: ['1' '2' '0']
Shapefile carregado com sucesso. Total de 224 municípios.
Municípios mesclados com sucesso: 224 de 224 (100.0%)


C:\Users\Seplan\AppData\Local\Temp\ipykernel_10116\712342557.py:146: DeprecationWarning: *choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.choropleth_mapbox(


Mapa interativo salvo como 'mapa_interativo_3clusters_educacao_20250317.html'


In [10]:
import pandas as pd

# Dados do CSV (3 clusters)
data = {
    "Cluster": [0, 1, 2],
    "tx_distorcao_fundamental": [23.6, 14.501, 10.6169],
    "taxa_escolas_por_habitante": [0.66127, 0.49220, 0.72350],
    "ideb_ano_inicial": [4.5775, 5.5769, 6.0729],
    "Total_alfabetização": [76.1077, 79.5973, 73.2329],
    "Taxa de abandonos EF": [1.8561, 0.4455, 0.1797]
}

# Diagnóstico manual para 3 clusters
diagnostico = {
    0: {
        "Pontos Fortes": "- Nenhum indicador destaca-se positivamente",
        "Pontos Fracos": "- IDEB crítico (4,58)\n- Abandono elevado (1,86%)\n- Distorção alta (23,6%)",
        "Recomendações": "- Reforma pedagógica\n- Combate ao abandono\n- Correção de fluxo escolar"
    },
    1: {
        "Pontos Fortes": "- Alfabetização alta (79,6%)\n- Baixo abandono (0,45%)",
        "Pontos Fracos": "- IDEB moderado (5,58)\n- Baixa densidade de escolas (0,49)",
        "Recomendações": "- Expansão de infraestrutura\n- Capacitação docente"
    },
    2: {
        "Pontos Fortes": "- Melhor IDEB (6,07)\n- Menor abandono (0,18%)",
        "Pontos Fracos": "- Alfabetização baixa (73,23%)\n- Excesso de escolas (0,72)",
        "Recomendações": "- Investigar contradição alfabetização/IDEB\n- Reorganizar rede escolar"
    }
}

# Criar DataFrame
df = pd.DataFrame(data)
df["Pontos Fortes"] = df["Cluster"].map(lambda x: diagnostico[x]["Pontos Fortes"])
df["Pontos Fracos"] = df["Cluster"].map(lambda x: diagnostico[x]["Pontos Fracos"])
df["Recomendações"] = df["Cluster"].map(lambda x: diagnostico[x]["Recomendações"])

# Exportar para CSV
df.to_csv("diagnostico_clusters_3.csv", index=False)

print("DataFrame com 3 clusters salvo como 'diagnostico_clusters_3.csv'")

DataFrame com 3 clusters salvo como 'diagnostico_clusters_3.csv'
